# CS13 Cell-Type and Marker-Gene Rendering

Render cell-type highlights and continuous marker-gene expression over an accepted CS13 reference mesh.

This curated notebook targets the current Dynamo-free Spateo API. Edit the path/configuration cells for a new system before execution.


# CS13 TRN100k celltype and marker-gene plotting

Plot the original TRN100k cells over an **already accepted** CS13 mesh. This notebook never reconstructs or edits the mesh. Celltype panels use categorical colors; marker-gene panels use one continuous `magma` expression scale with a visible scalar bar.


## Review gate and paths

Keep all three booleans `False` while inspecting the notebook. After accepting a mesh, set `MESH_APPROVED=True`, select the desired plotting sections, and change `OUTPUT_DIR` to a new empty directory. Existing outputs are never overwritten.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re

from datetime import datetime

import anndata as ad
import numpy as np
import pandas as pd
import pyvista as pv
import scipy.sparse as sp

os.environ.setdefault("PYVISTA_OFF_SCREEN", "true")
pv.OFF_SCREEN = True
import spateo as st

print("spateo", st.__version__)
print("pyvista", pv.__version__)


In [ ]:
MESH_APPROVED = False
RUN_CELLTYPE_PLOTS = False
RUN_MARKER_PLOTS = False

RUN_ROOT = Path("/DATA/User/gaomohan/figures/figure2/b/cs13_trn100k_3d_v1")
INPUT_H5AD = RUN_ROOT / "sampling/CS13.trn100k.h5ad"
MESH_PATH = (
    RUN_ROOT / "models_reference_construct_surface_v1/cs13_trn100k_mesh_reference_profile.vtk"
)
OUTPUT_DIR = RUN_ROOT / "plots_reference_mesh_v1"  # Change for every new plotting run.

SPATIAL_KEY = "spatial"
CELLTYPE_KEY = "celltype"
EXPRESSION_LAYER = None  # None uses adata.X; set 'counts' only when raw-count plotting is intended.
POSITIVE_CLIP_QUANTILE = 0.99
MARKERS_TO_PLOT = ["MYH6", "PAX6", "MYF5", "AFP", "EMCN", "EYA1", "HBE1", "HOXB9"]
CELLTYPES_TO_HIGHLIGHT = None  # None plots every observed celltype.

REFERENCE_CPO = [
    (7235.672822135923, -9333.743725966886, 29537.530999873827),
    (5265.3827, 317.52565000000004, 1200.0),
    (-0.9910735612049205, -0.13108990476291582, 0.024261763123203748),
]
REFERENCE_BACKGROUND = np.array([247, 248, 248], dtype=float) / 255.0

if (RUN_CELLTYPE_PLOTS or RUN_MARKER_PLOTS) and not MESH_APPROVED:
    raise RuntimeError("Set MESH_APPROVED=True only after manually accepting MESH_PATH.")


## Input contract

Require the exact 100,000-cell TRN subset, authoritative celltype labels, original 3D spatial coordinates, and a non-empty mesh. The plotting output is initialized only when at least one `RUN_*` switch is enabled.


## Load and validate data


In [ ]:
for required_path in (INPUT_H5AD, MESH_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

adata = ad.read_h5ad(INPUT_H5AD)
assert adata.n_obs == 100000, adata.n_obs
assert adata.obs_names.is_unique
assert CELLTYPE_KEY in adata.obs
assert SPATIAL_KEY in adata.obsm
assert np.asarray(adata.obsm[SPATIAL_KEY]).shape == (adata.n_obs, 3)
if EXPRESSION_LAYER is not None and EXPRESSION_LAYER not in adata.layers:
    raise KeyError(f"Missing layer: {EXPRESSION_LAYER}")

mesh = st.tdr.read_model(str(MESH_PATH)).extract_surface().triangulate().clean()
assert mesh.n_points > 0 and mesh.n_cells > 0

if RUN_CELLTYPE_PLOTS or RUN_MARKER_PLOTS:
    if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
        raise FileExistsError(f"Refusing to overwrite non-empty output: {OUTPUT_DIR}")
    (OUTPUT_DIR / "celltype_original_model").mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "marker_gene_original_model").mkdir(parents=True, exist_ok=True)

print("AnnData:", adata.shape)
print("Mesh:", mesh.n_points, "points /", mesh.n_cells, "cells")
print("Celltypes:", sorted(adata.obs[CELLTYPE_KEY].astype(str).unique()))


## Build the original annotated point cloud

The point order must remain identical to AnnData. `tissue_rgba` is used only by celltype panels. Marker panels explicitly ignore it.


## Construct the point-cloud model


In [ ]:
pointcloud, celltype_palette = st.tdr.construct_pc(
    adata=adata,
    spatial_key=SPATIAL_KEY,
    groupby=CELLTYPE_KEY,
    key_added="tissue",
    colormap="rainbow",
)
assert pointcloud.n_points == adata.n_obs
assert np.array_equal(
    np.asarray(pointcloud.point_data["obs_index"]).astype(str),
    adata.obs_names.astype(str).to_numpy(),
)
print("Point-cloud order verified:", pointcloud.n_points)


## Shared helpers


In [ ]:
def now_iso():
    return datetime.now().astimezone().isoformat(timespec="seconds")

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def safe_filename(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")

def expression_vector(adata, gene, layer=None):
    subset = adata[:, gene]
    values = subset.layers[layer] if layer is not None else subset.X
    if sp.issparse(values):
        values = values.toarray()
    return np.asarray(values, dtype=float).reshape(-1)

def set_reference_camera(plotter):
    plotter.camera_position = REFERENCE_CPO
    plotter.reset_camera_clipping_range()


# Celltype plotting

The overview displays categorical colors for all celltypes. Each highlight keeps the selected celltype in its categorical color and renders all non-target cells faint grey.


In [ ]:
def render_all_celltypes(mesh, pointcloud, output):
    plotter = pv.Plotter(off_screen=True, window_size=(1200, 1200))
    plotter.set_background(REFERENCE_BACKGROUND)
    plotter.add_mesh(
        mesh,
        color="gainsboro",
        opacity=0.12,
        smooth_shading=True,
        ambient=0.35,
        show_scalar_bar=False,
    )
    plotter.add_mesh(
        pointcloud,
        scalars="tissue_rgba",
        rgba=True,
        style="points",
        model_size=1.4,
        opacity=1.0,
        render_points_as_spheres=False,
        show_scalar_bar=False,
    )
    labels = np.asarray(pointcloud.point_data["tissue"]).astype(str)
    rgba = np.asarray(pointcloud.point_data["tissue_rgba"], dtype=float)
    legend_entries = []
    for label in sorted(np.unique(labels)):
        idx = int(np.flatnonzero(labels == label)[0])
        legend_entries.append([label, tuple(rgba[idx, :3])])
    legend = plotter.add_legend(
        legend_entries,
        bcolor=REFERENCE_BACKGROUND,
        face="circle",
        loc="upper right",
        size=(0.20, 0.42),
    )
    legend.GetEntryTextProperty().SetFontSize(8)
    legend.SetPosition(0.78, 0.55)
    legend.SetPosition2(0.20, 0.42)
    plotter.add_text("CS13 cell types", position="upper_left", font_size=14, color="black")
    set_reference_camera(plotter)
    plotter.screenshot(str(output), transparent_background=True)
    plotter.close()

def render_celltype_highlight(mesh, pointcloud, labels, celltype, output):
    selected = labels == celltype
    other_pc = pointcloud.extract_points(~selected, adjacent_cells=False)
    selected_pc = pointcloud.extract_points(selected, adjacent_cells=False)
    other_pc.point_data["tissue_rgba"] = np.tile(
        np.array([[0.55, 0.55, 0.55, 1.0]]), (other_pc.n_points, 1)
    )
    plotter = pv.Plotter(off_screen=True, window_size=(1200, 1200))
    plotter.set_background(REFERENCE_BACKGROUND)
    plotter.add_mesh(
        mesh,
        color="gainsboro",
        opacity=0.12,
        smooth_shading=True,
        ambient=0.35,
        show_scalar_bar=False,
    )
    plotter.add_mesh(
        other_pc,
        scalars="tissue_rgba",
        rgba=True,
        style="points",
        model_size=1.0,
        opacity=0.06,
        render_points_as_spheres=False,
        show_scalar_bar=False,
    )
    plotter.add_mesh(
        selected_pc,
        scalars="tissue_rgba",
        rgba=True,
        style="points",
        model_size=2.0,
        opacity=1.0,
        render_points_as_spheres=False,
        show_scalar_bar=False,
    )
    plotter.add_text(celltype, position="upper_left", font_size=14, color="black")
    set_reference_camera(plotter)
    plotter.screenshot(str(output), transparent_background=True)
    plotter.close()


In [ ]:
celltype_outputs = []
if RUN_CELLTYPE_PLOTS:
    celltype_dir = OUTPUT_DIR / "celltype_original_model"
    overview = celltype_dir / "cs13_all_celltypes_original_model.png"
    render_all_celltypes(mesh, pointcloud, overview)
    celltype_outputs.append(str(overview))

    labels = adata.obs[CELLTYPE_KEY].astype(str).to_numpy()
    observed = sorted(np.unique(labels))
    selected_celltypes = observed if CELLTYPES_TO_HIGHLIGHT is None else CELLTYPES_TO_HIGHLIGHT
    for celltype in selected_celltypes:
        if celltype not in observed:
            print("Skipping absent celltype:", celltype)
            continue
        output = celltype_dir / f"celltype_{safe_filename(celltype)}_original_model.png"
        render_celltype_highlight(mesh, pointcloud, labels, celltype, output)
        celltype_outputs.append(str(output))

print("Celltype outputs:", len(celltype_outputs))


# Marker-gene selection and specificity

The default panel contains eight spatially informative CS13 markers. The specificity table evaluates additional candidates against their expected target celltypes before plotting.


In [ ]:
MARKER_CANDIDATES = {
    "Heart": ["MYH6", "TNNT2", "ACTC1", "NKX2-5"],
    "Neural": ["PAX6", "SOX2", "SOX1", "DCX"],
    "Myotome": ["MYF5", "MYOG", "MYOD1", "DES"],
    "Liver_or_yolk_sac_endoderm": ["AFP", "APOA2", "ALB", "HNF4A"],
    "VEC": ["EMCN", "CDH5", "KDR", "PECAM1"],
    "Sensory_organ_primordium": ["EYA1", "SIX1", "PAX2", "PAX8"],
    "Erythroid": ["HBE1", "GYPA", "KLF1", "HBB"],
    "tail_bud": ["HOXB9", "CDX2", "TBXT", "WNT3A"],
}
TARGET_ALIASES = {
    "Liver_or_yolk_sac_endoderm": ["Liver", "YS.Endo"],
    "Erythroid": ["Ery1", "Ery2"],
}

def marker_specificity_table(adata):
    labels = adata.obs[CELLTYPE_KEY].astype(str).to_numpy()
    available = set(adata.var_names.astype(str))
    rows = []
    for target, genes in MARKER_CANDIDATES.items():
        target_names = TARGET_ALIASES.get(target, [target])
        target_mask = np.isin(labels, target_names)
        for gene in genes:
            row = {"target": target, "gene": gene, "available": gene in available}
            if gene in available:
                values = expression_vector(adata, gene, EXPRESSION_LAYER)
                other_mask = ~target_mask
                target_mean = float(values[target_mask].mean()) if target_mask.any() else 0.0
                other_mean = float(values[other_mask].mean()) if other_mask.any() else 0.0
                row.update(
                    {
                        "target_mean": target_mean,
                        "other_mean": other_mean,
                        "target_detect_fraction": float(np.mean(values[target_mask] > 0))
                        if target_mask.any()
                        else 0.0,
                        "other_detect_fraction": float(np.mean(values[other_mask] > 0))
                        if other_mask.any()
                        else 0.0,
                        "mean_specificity_ratio": float((target_mean + 1e-6) / (other_mean + 1e-6)),
                        "selected_for_plot": gene in MARKERS_TO_PLOT,
                    }
                )
            rows.append(row)
    return pd.DataFrame(rows)

marker_specificity = marker_specificity_table(adata)
marker_specificity.sort_values(["target", "mean_specificity_ratio"], ascending=[True, False])


# Marker-gene plotting

Only `expression` is passed to PyVista as the scalar. `tissue_rgba` is never used in these panels. Therefore multiple colors represent different values on the single continuous `magma` scale—not different celltypes. Zero-expression cells are nearly transparent; positive cells use normalized expression for both color and opacity.


In [ ]:
def render_marker_gene(mesh, pointcloud, values, gene, output, positive_clip_quantile=0.99):
    positive = values[values > 0]
    if positive.size == 0:
        return None
    clip_value = max(
        float(np.quantile(positive, positive_clip_quantile)),
        float(positive.min()),
        1e-12,
    )
    normalized = np.clip(values / clip_value, 0.0, 1.0).astype(np.float32)
    expression_alpha = np.where(values > 0, np.maximum(0.08, normalized), 0.025).astype(np.float32)
    marker_pc = pointcloud.copy()
    marker_pc.point_data["expression"] = normalized
    marker_pc.point_data["expression_alpha"] = expression_alpha

    plotter = pv.Plotter(off_screen=True, window_size=(1200, 1200))
    plotter.set_background("white")
    plotter.add_mesh(
        mesh,
        color="gainsboro",
        opacity=0.05,
        smooth_shading=True,
        ambient=0.35,
        show_scalar_bar=False,
    )
    plotter.add_mesh(
        marker_pc,
        scalars="expression",
        cmap="magma",
        clim=(0.0, 1.0),
        opacity="expression_alpha",
        style="points",
        model_size=1.6,
        render_points_as_spheres=False,
        show_scalar_bar=True,
        scalar_bar_args={
            "title": f"{gene} normalized expression\n(positive p99 = 1)",
            "vertical": True,
            "position_x": 0.85,
            "position_y": 0.20,
            "height": 0.55,
            "width": 0.08,
            "title_font_size": 10,
            "label_font_size": 9,
            "n_labels": 5,
            "fmt": "%.2f",
        },
    )
    plotter.add_text(gene, position="upper_left", font_size=14, color="black")
    set_reference_camera(plotter)
    plotter.screenshot(str(output), transparent_background=False)
    plotter.close()
    return {
        "gene": gene,
        "output": str(output),
        "positive_cells": int(np.sum(values > 0)),
        "positive_fraction": float(np.mean(values > 0)),
        "positive_clip_quantile": positive_clip_quantile,
        "clip_value": clip_value,
        "expression_source": "adata.X"
        if EXPRESSION_LAYER is None
        else f"adata.layers[{EXPRESSION_LAYER!r}]",
        "color_encoding": "single continuous magma normalized expression; no celltype colors",
        "opacity_encoding": "zero alpha=0.025; positive alpha=max(0.08, normalized expression)",
    }


In [ ]:
marker_outputs = []
if RUN_MARKER_PLOTS:
    marker_dir = OUTPUT_DIR / "marker_gene_original_model"
    available = set(adata.var_names.astype(str))
    marker_specificity.to_csv(OUTPUT_DIR / "marker_gene_specificity.csv", index=False)
    for gene in MARKERS_TO_PLOT:
        if gene not in available:
            print("Skipping absent gene:", gene)
            continue
        values = expression_vector(adata, gene, EXPRESSION_LAYER)
        output = marker_dir / f"marker_{safe_filename(gene)}_original_model.png"
        record = render_marker_gene(mesh, pointcloud, values, gene, output, POSITIVE_CLIP_QUANTILE)
        if record is None:
            print("Skipping unexpressed gene:", gene)
        else:
            marker_outputs.append(record)

print("Marker outputs:", len(marker_outputs))


## Provenance manifest

Write one manifest only after at least one plotting section has run successfully.


In [ ]:
if RUN_CELLTYPE_PLOTS or RUN_MARKER_PLOTS:
    manifest = {
        "version": "cs13_trn100k_celltype_marker_notebook_v1",
        "created_at": now_iso(),
        "status": "PASS",
        "mesh_approved": MESH_APPROVED,
        "input_h5ad": str(INPUT_H5AD),
        "input_h5ad_sha256": sha256(INPUT_H5AD),
        "mesh": str(MESH_PATH),
        "mesh_sha256": sha256(MESH_PATH),
        "n_points": int(adata.n_obs),
        "celltype_key": CELLTYPE_KEY,
        "spatial_key": SPATIAL_KEY,
        "expression_layer": EXPRESSION_LAYER,
        "camera_position": REFERENCE_CPO,
        "celltype_outputs": celltype_outputs,
        "marker_outputs": marker_outputs,
        "marker_specificity_csv": (
            str(OUTPUT_DIR / "marker_gene_specificity.csv") if RUN_MARKER_PLOTS else None
        ),
        "color_contract": {
            "celltype": "categorical tissue_rgba from obs[celltype]",
            "marker_gene": "single continuous magma normalized expression; tissue_rgba is ignored",
        },
        "background_contract": {
            "celltype": "transparent background with reference RGB 247/248/248",
            "marker_gene": "opaque white background",
        },
    }
    manifest_path = OUTPUT_DIR / "plot_manifest.json"
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n")
    print(manifest_path)
